# Session 13 — Run and preserve the complete experiment

Select **Runtime → Change runtime type → GPU**, then run all cells. This notebook executes the actual core notebooks and saves their outputs automatically. The required experiments consume **150M successful-update tokens**, plus two 2M-token variant pilots.

Checkpoints are saved every 500 successful optimizer updates. Google Drive persistence is enabled below so a disconnected runtime can resume. The same GPU type, software versions, precision, source and data must be used throughout the comparison.


In [ ]:
from pathlib import Path
import os, subprocess, sys

PERSIST_TO_DRIVE = True  # @param {type:"boolean"}
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/drive/MyDrive/ERA_V5_S13/repository')
else:
    REPO_DIR = Path('/content/era-v5-session-13-reversible-llm-lab')
REPO_URL = 'https://github.com/JoeIndyGit/era-v5-session-13-reversible-llm-lab.git'
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    # Keep the exact source for an experiment already in progress.
    evidence = list((REPO_DIR / 'results').glob('*.json')) + list((REPO_DIR / 'checkpoints').glob('*.pt'))
    if not evidence:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print('Persistent experiment folder:', REPO_DIR)
print('Source commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → GPU, then reconnect.'
print(torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GiB')
subprocess.run(['nvidia-smi'], check=True)


The default fixed batch is 16. Precision is selected from GPU support, then validated. If the numerical gate rejects both variants, use FP32 **in a fresh experiment folder** for all three comparisons. Changing precision halfway through an experiment invalidates the controlled comparison.

Running this cell again resumes incomplete experiments and verifies completed results before reusing them. Live progress is printed below; notebook outputs are also saved under `executed_notebooks/`.


In [ ]:
PRECISION = 'auto'  # @param ['auto', 'fp32', 'fp16', 'bf16']
FIXED_BATCH = 16  # @param {type:"integer"}
subprocess.run([sys.executable, 'scripts/run_full_submission.py', '--precision', PRECISION, '--batch-size', str(FIXED_BATCH)], check=True)


In [ ]:
import json
manifest = json.loads(Path('submission_evidence/MANIFEST.json').read_text())
print('Evidence audit:', manifest['audit'])
print('Packaged files:', len(manifest['files']))
print('Evidence ZIP:', REPO_DIR / 'submission_evidence/era-v5-session-13-evidence.zip')


## Submit the measured output

The console must show `FINAL EVIDENCE AUDIT: PASS` and `COMPLETE`. Commit the generated `README.md`, `results/`, `assets/`, and `executed_notebooks/` to the repository. The evidence ZIP contains these files plus the source and tokenizer needed to inspect the experiment. Checkpoints remain in the persistent experiment folder and are intentionally excluded from GitHub.

The final report discusses held-out quality, same-batch memory savings, throughput cost, maximum-batch uplift, reconstruction error, overflow retries, and recovery. A lower speed or worse loss remains a valid finding; report the measurements as obtained.
